# Pipeline d'Analyse PDF à Grande Échelle
## Étape 01 : Extraction RAW des Bookmarks / Table of Contents (TOC)

---

### 🎯 Objectif de cette étape (Couche RAW)
Cette étape constitue le **socle déterministe et ultra-rapide** du pipeline de traitement de documents PDF techniques volumineux (pouvant atteindre plusieurs milliers de pages).

Elle a pour mission exclusive d'extraire la structure interne des signets (*bookmarks / outline / TOC*) telle que définie dans le fichier PDF, **sans aucune perte d'information ni altération**.

---

### 🛡️ Principes Directeurs et Contraintes Strictes :
1. **Extraction RAW pure** : aucune interprétation métier, aucune heuristique.
2. **Zéro suppression** : aucune entrée n'est retirée (même si `page == -1`, titre vide, doublon, langue ENG/FRE, niveau profond, etc.).
3. **Zéro déduplication / Zéro regroupement** : l'ordre exact et la position d'origine (`index`) sont préservés à 100%.
4. **Performance maximale** : utilise uniquement `PyMuPDF` (`fitz`) et `doc.get_toc(simple=True)` (exécution en quelques millisecondes, sans OCR, sans LLM, sans analyse visuelle des pages).
5. **Gestion des PDF sans TOC** : si `get_toc()` retourne `[]`, produit proprement `sections: []` et `toc_available: false` sans inventer de données.


---
## 1. Installation des Dépendances
Nous utilisons **PyMuPDF (`fitz`)**, la bibliothèque la plus rapide et robuste pour l'accès instantané aux métadonnées et à la table des matières interne des fichiers PDF.


In [ ]:
# Installation de PyMuPDF si nécessaire
%pip install -q pymupdf

import fitz
print(f"[OK] PyMuPDF (fitz) version : {fitz.__version__}")


---
## 2. Imports & Configuration de l'Environnement


In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Dict, List, Any, Optional, Union

print("[OK] Modules standards importes avec succes.")


---
## 3. Configuration des Chemins (Input / Output)

Le pipeline supporte :
- Un fichier PDF unique
- Un dictionnaire de fichiers PDF cibles
- Un dossier entier contenant des fichiers PDF


In [ ]:
# Dossier contenant les PDF d'entree
DATA_DIR = Path("data")

# Dossier ou seront enregistres les fichiers JSON RAW extraits
OUTPUT_DIR = Path(".")

# Dictionnaire de fichiers PDF cibles
PDFS: Dict[str, Path] = {
    "AUSTCOLD": DATA_DIR / "AUSTCOLD.pdf",
    "MYCOM": DATA_DIR / "MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf"
}

# Auto-decouverte des autres PDF dans DATA_DIR s'ils existent
if DATA_DIR.exists():
    for pdf_file in DATA_DIR.glob("*.pdf"):
        key = pdf_file.stem.split()[0]
        if key not in PDFS:
            PDFS[key] = pdf_file

print("Fichiers PDF configures pour extraction :")
for name, path in PDFS.items():
    status = "[TROUVE]" if path.exists() else "[INTROUVABLE]"
    print(f"  * {name:<12} : {path.name} {status}")


---
## 4. Fonction d'Extraction Déterministe : `extract_raw_toc()`

Cette fonction ouvre le document de manière sécurisée et extrait :
- Les métadonnées globales (`filename`, `total_pages`, `toc_available`, `toc_entries_count`)
- La liste complète et ordonnée des entrées avec leur `index` original (0-indexed), `level`, `title` et `page`.


In [ ]:
def extract_raw_toc(pdf_path: Union[str, Path]) -> Dict[str, Any]:
    """
    Extrait de maniere exhaustive, deterministe et sans alteration 
    la table des matieres interne (bookmarks/TOC) d'un document PDF.
    
    Args:
        pdf_path: Chemin vers le fichier PDF.
        
    Returns:
        Dictionnaire contenant les metadonnees du document et la liste RAW des sections.
    """
    path = Path(pdf_path)
    
    # 1. Verification de l'existence du fichier
    if not path.exists():
        raise FileNotFoundError(f"Le fichier PDF est introuvable : {path.resolve()}")
    
    # 2. Ouverture securisee avec PyMuPDF
    try:
        with fitz.open(str(path)) as doc:
            if doc.is_encrypted:
                raise PermissionError(f"Le fichier '{path.name}' est protege / chiffre par mot de passe.")
            
            total_pages = len(doc)
            
            # Extraction pure des bookmarks natifs du PDF
            # Format retourne par doc.get_toc(simple=True) : [[level: int, title: str, page: int], ...]
            raw_toc = doc.get_toc(simple=True)
            
            toc_entries_count = len(raw_toc)
            toc_available = (toc_entries_count > 0)
            
            # Construction de la liste des sections avec preservation stricte de l'ordre et des valeurs
            sections = [
                {
                    "index": idx,
                    "level": int(entry[0]),
                    "title": str(entry[1]),
                    "page": int(entry[2])
                }
                for idx, entry in enumerate(raw_toc)
            ]
            
            # Structure de sortie standardisee
            output_data = {
                "document": {
                    "filename": path.name,
                    "total_pages": total_pages,
                    "toc_available": toc_available,
                    "toc_entries_count": toc_entries_count
                },
                "sections": sections
            }
            
            return output_data
            
    except Exception as e:
        raise RuntimeError(f"Erreur lors de la lecture du document '{path.name}' avec PyMuPDF : {e}") from e

print("[OK] Fonction extract_raw_toc() definie et prete.")


---
## 5. Fonction de Sauvegarde JSON & Synthèse Visuelle


In [ ]:
def process_and_save_pdf(
    pdf_path: Union[str, Path], 
    output_dir: Union[str, Path] = ".",
    custom_name: Optional[str] = None,
    suffix: str = "_sections_raw.json"
) -> Dict[str, Any]:
    """
    Extrait le TOC RAW d'un PDF, l'enregistre au format JSON (UTF-8)
    et affiche un resume clair et lisible.
    """
    pdf_p = Path(pdf_path)
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Extraction RAW
    data = extract_raw_toc(pdf_p)
    
    # 2. Determination du chemin de sortie : <STEM>_sections_raw.json
    stem = custom_name if custom_name else pdf_p.stem
    output_filename = f"{stem}{suffix}"
    output_path = out_dir / output_filename
    
    # 3. Ecriture du fichier JSON (UTF-8 avec indent=2 et conservation des accents francais)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    # 4. Affichage du resume formate
    doc_info = data["document"]
    print("=" * 50)
    print(f"{doc_info['filename']}")
    print("=" * 50)
    print(f"Total pages     : {doc_info['total_pages']}")
    print(f"TOC available   : {doc_info['toc_available']}")
    print(f"TOC entries     : {doc_info['toc_entries_count']}")
    print(f"Output          : {output_path.name}")
    print()
    
    return data

print("[OK] Fonction process_and_save_pdf() prete.")


---
## 6. Exécution sur un PDF Unique (Exemple : AUSTCOLD.pdf)


In [ ]:
# Traitement d'un PDF individuel
target_pdf = PDFS.get("AUSTCOLD", DATA_DIR / "AUSTCOLD.pdf")

if target_pdf.exists():
    austcold_data = process_and_save_pdf(target_pdf, output_dir=OUTPUT_DIR, custom_name="AUSTCOLD")
    
    # Apercu des 6 premieres entrees RAW extraites
    print("Apercu des premieres entrees RAW extraites :")
    print(json.dumps(austcold_data["sections"][:6], ensure_ascii=False, indent=2))
else:
    print(f"[ATTENTION] Le fichier {target_pdf} n'existe pas dans le repertoire.")


---
## 7. Traitement Multi-PDFs (Mode Batch)


In [ ]:
# Traitement de l'ensemble des fichiers configures
batch_results = {}

for label, pdf_path in PDFS.items():
    if pdf_path.exists():
        data = process_and_save_pdf(pdf_path, output_dir=OUTPUT_DIR, custom_name=label)
        batch_results[label] = data
    else:
        print(f"[SKIP] Fichier absent : {pdf_path}")

print(f"[OK] Traitement batch termine. {len(batch_results)} fichier(s) JSON RAW genere(s).")


---
## 8. Assertions & Validation de Fidélité Déterministe

Cette cellule valide rigoureusement que :
1. `len(output["sections"]) == len(doc.get_toc(simple=True))`
2. L'ordre et les indices sont strictement séquentiels (`0, 1, ..., N-1`).
3. Chaque champ (`level`, `title`, `page`) correspond exactement à l'entrée PyMuPDF sans altération.
4. Les pages spéciales comme `page = -1` sont intégralement conservées.
5. Les PDF sans TOC (`MYCOM.pdf`) produisent `sections = []` et `toc_available = False`.
6. La sérialisation JSON respecte l'encodage UTF-8 et la structure attendue.


In [ ]:
# Execution de la suite de tests de validation sur chaque PDF traite
print("Execution des verifications d'integrite...")

for label, data in batch_results.items():
    pdf_path = PDFS[label]
    
    with fitz.open(pdf_path) as doc:
        original_toc = doc.get_toc(simple=True)
        sections = data["sections"]
        doc_info = data["document"]
        
        # Test 1 : Egalite exacte du nombre d'entrees
        assert len(sections) == len(original_toc), (
            f"[{label}] Incoherence de taille : JSON={len(sections)} vs PyMuPDF={len(original_toc)}"
        )
        assert doc_info["toc_entries_count"] == len(original_toc), (
            f"[{label}] Incoherence toc_entries_count : {doc_info['toc_entries_count']} vs {len(original_toc)}"
        )
        assert doc_info["total_pages"] == len(doc), (
            f"[{label}] Incoherence total_pages : {doc_info['total_pages']} vs {len(doc)}"
        )
        assert doc_info["toc_available"] == (len(original_toc) > 0), (
            f"[{label}] Incoherence toc_available : {doc_info['toc_available']}"
        )
        
        # Test 2 : Fidelite 1:1 de chaque entree
        for i, (orig_entry, json_sec) in enumerate(zip(original_toc, sections)):
            assert json_sec["index"] == i, f"[{label}] Index non sequentiel a la position {i}"
            assert json_sec["level"] == orig_entry[0], f"[{label}] Erreur level a l'index {i}"
            assert json_sec["title"] == orig_entry[1], f"[{label}] Erreur title a l'index {i}"
            assert json_sec["page"] == orig_entry[2], f"[{label}] Erreur page a l'index {i}"
        
        # Test 3 : Verification specifique des entrees a page -1 si presentes
        neg_pages_orig = [e for e in original_toc if e[2] == -1]
        neg_pages_json = [s for s in sections if s["page"] == -1]
        assert len(neg_pages_orig) == len(neg_pages_json), (
            f"[{label}] Incoherence sur les pages a -1 : {len(neg_pages_json)} vs {len(neg_pages_orig)}"
        )
        
        print(f"  [VALIDE] [{label:<12}] : {len(sections)} entrees verifiees a 100% avec l'original.")

print("\n[SUCCES] TOUTES LES ASSERTIONS ONT REUSSI : L'extraction RAW est 100% fidele et sans perte.")
